In [0]:
def populateConsumerSubscription(df_CycleData_ConsumerSubscription_Live,UpdatedBy="ADB_UserSubscription"):

    # Deduplicating data by taking only minimum cycle
    Window_Deduplicate = Window.partitionBy('CoolSculptingID','SoldToAccountID','PromotionUUID','ConsumerSubscriptionUUID').orderBy(asc('CycleDate'),asc('MinCycleTmstmp'),asc('ShipToAccountUUID'))

    df_CycleData_AGG_CycleClassification_DeDUP = (df_CycleData_ConsumerSubscription_Live
                                                .withColumn('rownum',row_number().over(Window_Deduplicate))
                                                .filter('rownum = 1 AND NewSubscriptionFlag = 1')
                                                .drop('rownum')
                                                )

    df_ConsumerSubscription_New = applyRules(df_CycleData_AGG_CycleClassification_DeDUP,'ConsumerSubscription_Live')

    df_ConsumerSubscription_New = (df_ConsumerSubscription_New
                                    .withColumn('VersionID',lit(1))
                                    .withColumn('NewConsumerSubscriptionUUID',expr('uuid()'))
                                    .withColumn('ConsumerSubscriptionUUID',lit(None).cast(StringType()))
                                    .withColumn('CreatedBy',lit(UpdatedBy))
                                    .withColumn('CreatedDate',lit(current_timestamp()))
                                    .withColumn('UpdatedBy',lit(UpdatedBy))
                                    .withColumn('UpdatedDate',lit(current_timestamp()))
                                    )

    dl_FACTConsumer = DeltaTable.forName(spark, 'Promotion.FACT_ConsumerSubscription')
    (dl_FACTConsumer.alias("tgt")
        .merge(df_ConsumerSubscription_New.alias("src"),
                ("tgt.CoolSculptingID = src.CoolSculptingID AND (tgt.ShipToAccountUUID = src.ShipToAccountUUID OR tgt.SoldToAccountID = src.SoldToAccountID) AND tgt.PromotionUUID = src.PromotionUUID AND src.NewSubscriptionStartDate BETWEEN tgt.SubscriptionStartDate AND tgt.SubscriptionEndDate  AND tgt.VersionID = src.VersionID"))
        .whenNotMatchedInsert(values =
        {
        "tgt.ConsumerSubscriptionUUID": "src.NewConsumerSubscriptionUUID",
        "tgt.CoolSculptingID": "src.CoolSculptingID",
        "tgt.ShipToAccountUUID": "src.ShipToAccountUUID",
        "tgt.SoldToAccountID": "src.SoldToAccountID",
        "tgt.PromotionUUID": "src.PromotionUUID",            
        "tgt.SubscriptionStartDate": "src.NewSubscriptionStartDate",
        "tgt.SubscriptionEndDate": "src.NewSubscriptionEndDate",
        "tgt.InvoiceExceptionFlag": "src.InvoiceExceptionFlag",        
        "tgt.VersionID": "src.VersionID",
        "tgt.Comments": "src.Comments",
        "tgt.CreatedBy": "src.CreatedBy",
        "tgt.CreatedDate": "src.CreatedDate",
        "tgt.UpdatedBy": "src.UpdatedBy",
        "tgt.UpdatedDate": "src.UpdatedDate"
        }
        )
        .execute())


In [0]:
def populateConsumerSubscription_ConsumerBased(df_CycleData_ConsumerSubscription_Live,UpdatedBy="ADB_UserSubscription"):

    # Deduplicating data by taking only minimum cycle
    Window_Deduplicate = Window.partitionBy('CoolSculptingID','PromotionUUID','ConsumerSubscriptionUUID').orderBy(asc('CycleDate'),asc('MinCycleTmstmp'))

    df_CycleData_AGG_CycleClassification_DeDUP = (df_CycleData_ConsumerSubscription_Live
                                                    .filter('NewSubscriptionFlag = 1')
                                                    .withColumn('rownum',row_number().over(Window_Deduplicate))
                                                    .filter('rownum = 1')
                                                    .drop('rownum')
                                                    )

    df_ConsumerSubscription_New = applyRules(df_CycleData_AGG_CycleClassification_DeDUP,'ConsumerSubscription_Live')

    df_ConsumerSubscription_New = (df_ConsumerSubscription_New
                                    .withColumn('VersionID',lit(1))
                                    .withColumn('NewConsumerSubscriptionUUID',expr('uuid()'))
                                    .withColumn('ConsumerSubscriptionUUID',lit(None).cast(StringType()))
                                    .withColumn('CreatedBy',lit(UpdatedBy))
                                    .withColumn('CreatedDate',lit(current_timestamp()))
                                    .withColumn('UpdatedBy',lit(UpdatedBy))
                                    .withColumn('UpdatedDate',lit(current_timestamp()))
                                    )

    dl_FACTConsumer = DeltaTable.forName(spark, 'Promotion.FACT_ConsumerSubscription')
    (dl_FACTConsumer.alias("tgt")
        .merge(df_ConsumerSubscription_New.alias("src"),
                ("tgt.CoolSculptingID = src.CoolSculptingID AND tgt.PromotionUUID = src.PromotionUUID AND src.NewSubscriptionStartDate BETWEEN tgt.SubscriptionStartDate AND tgt.SubscriptionEndDate  AND tgt.VersionID = src.VersionID"))
        .whenNotMatchedInsert(values =
        {
        "tgt.ConsumerSubscriptionUUID": "src.NewConsumerSubscriptionUUID",
        "tgt.CoolSculptingID": "src.CoolSculptingID",
        "tgt.ShipToAccountUUID": "src.ShipToAccountUUID",
        "tgt.SoldToAccountID": "src.SoldToAccountID",
        "tgt.PromotionUUID": "src.PromotionUUID",            
        "tgt.SubscriptionStartDate": "src.NewSubscriptionStartDate",
        "tgt.SubscriptionEndDate": "src.NewSubscriptionEndDate",
        "tgt.InvoiceExceptionFlag": "src.InvoiceExceptionFlag",        
        "tgt.VersionID": "src.VersionID",
        "tgt.Comments": "src.Comments",
        "tgt.CreatedBy": "src.CreatedBy",
        "tgt.CreatedDate": "src.CreatedDate",
        "tgt.UpdatedBy": "src.UpdatedBy",
        "tgt.UpdatedDate": "src.UpdatedDate"
        }
        )
        .execute())
